# F-тест: значимость регрессии

Тестируется гипотеза $H_0:\beta_1=\cdots=\beta_k=0$

Тестовая статистика $F$ (расчитывается автоматически)

Критическое значение $F_{cr}=F_{df1=k, df2=n-k-1}(\alpha)$

Гипотеза отвергается, если $F>F_{cr}$ или $P<\alpha$

# LPM-модель: F-тест на значимость регрессии

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import f # F-распределение

### approve equation #1

Для датасета `loanapp` рассморим регрессию **approve на unem, male, yjob, self**

Спецификация: 
$$
  approve=\beta_0+\beta_1unem+\beta_2male+\beta_3yjob+\beta_4self+u
$$

Альтернативная спецификация:
$$
  P(approve=1)=\beta_0+\beta_1unem+\beta_2male+\beta_3yjob+\beta_4self
$$

In [ ]:
# загрузим данные
df = pd.read_csv('../../datasets/loanapp.csv')

In [ ]:
#зададим спецификацию модели через формулу
mod = smf.ols(formula='approve ~ 1 +unem + male + yjob + self', data=df)

In [ ]:
# подгонка модели с поправкой на гетероскедастичность
res_hc = mod.fit(cov_type='HC3')
print(res_hc.summary(slim=True))

### Тестируем значимость регрессии, т.е. гипотезу $$H_0:\beta_{unem}=\beta_{male}=\beta_{yjob}=\beta_{self}=0.$$

### Результаты робастного F-теста

In [ ]:
# Тестовая статистика
round(res_hc.fvalue, 3)

In [ ]:
# P-значение тестовой статистики
round(res_hc.f_pvalue, 3)

### Результаты неробастного F-теста

In [ ]:
# подгонка модели
res_ols = mod.fit(cov_type='nonrobust')

# тестовая статистика и P-значение
round(res_ols.fvalue, 3), round(res_ols.f_pvalue, 3)

### 10\%-критическое значение F-распределения

In [ ]:
# зададим уровень значимости
sign_level = 0.1
# 10%-критическое значение F-распределения
f.isf(sign_level, dfn=mod.df_model, dfd=mod.df_resid).round(3)

## **Вывод:** регрессия незначима!

### Альтернативный способ: явно специфицируем тестируемую гипотезу

In [ ]:
# тестовая статистика, P-значение и степени свободы
F_test = res_hc.wald_test('unem=0, male=0, yjob=0, self=0', use_f=True)
print(F_test)